In [ ]:
'''
abbreviation notes: TDE (tidal disruption event), TC (TDE catalog), PSC (post-starburst catalog),
    SMBHB (supermassive black hole binary)
'''

In [3]:
import os
import otter

from astropy.coordinates import SkyCoord
from astropy import units as u

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [4]:
db = otter.Otter()

Attempting to login to https://otter.idies.jhu.edu/api with the following credentials:
username: user-guest
password: test


In [15]:
'''
initial function
returns list of potential galaxy canidates
'''
# note: only concerned with TC for now

# initialize canidate list as objects classified as TDEs and within redshift 0.03
cands = db.query(classification="TDE", minz=0, maxz=0.03, class_confidence_threshold=1)

# initalize lists to be stored in dataframe
name_list = []

rs_list = []
lum_list = []

cord_ra_list = []
cord_dec_list = []

confidence = []

host_names = []
hosts_cords = []

# iterate through all transients returned by initial query
i = 0
while i < len(cands):
    object = cands[i]
    
    name = object.default_name
    name_list.append(name)

    
    dists = pd.DataFrame(object['distance']) # find all possible distance measurments
    # luminosity
    lum_dists = dists[dists.distance_type == 'luminosity']
    if len(lum_dists) != 0:
        obj_lums = lum_dists['value'].tolist()
    else:
        obj_lum = 'Empty'
    lum_list.append(obj_lums)
    # redshift
    rs_dists = dists[dists.distance_type == 'redshift']
    obj_rs = (rs_dists['value']).tolist()
    rs_list.append(obj_rs)

    # ra and dec of transient in deg
    cords = object.get_skycoord()
    ra = cords.ra.deg
    cord_ra_list.append(ra)
    dec = cords.dec.deg
    cord_dec_list.append(dec)
    
    # confidence transient is a TDE
    conf = object.get_classification()
    confidence.append(conf[1]) # filter for confidences > 1
    
    info = object.get_host() # does OTTER have an object of class host assosiated with the transient?
    # need to know this to know if OTTER can return photometric information
    if info == []:
        host_names.append('Unknown!')
        hosts_cords.append('N/A')
    else:
        host_names.append(info[0].name)
        hosts_cords.append((info[0].coord.ra.deg, info[0].coord.dec.deg))

    i += 1


# initialize data frame storing all possible canidates
cands_df = pd.DataFrame(
    {"TDE Name": name_list, "default redshift": [item[0] for item in rs_list], "redshifts": rs_list, "luminosity (Mpc)": lum_list, 
     "confidence": confidence, "ra (deg)": cord_ra_list, "dec (deg)":cord_dec_list, "Host Name": host_names, 
     "Host Coords (deg)": hosts_cords}
)

cands_df = cands_df.set_index("TDE Name")

cands_df

,default redshift,redshifts,luminosity (Mpc),confidence,ra (deg),dec (deg),Host Name,Host Coords (deg)
TDE Name,,,,,,,,
NGC 5905,0.01124,"[0.01124, 0.012]",[50.17],3.1,228.847167,55.517108,NGC 5905,"(228.8458333333333, 55.51710833333333)"
ARP 299-B AT1,0.010411,[0.010411],[50.17],3.3,172.129115,58.561329,Unknown!,N/A
XMMSL1 J0740-85,0.0173,[0.0173],[77.6],3.3,115.034125,-85.658681,2MASX 07400785-8539307,"(115.03333333333333, 85.6575)"
XMMSL2 J144605.0+685735,0.029,[0.029],[77.6],3.3,221.520833,68.958611,"2MASX 14460522+6857311, LEDA 2725953","(221.52083333333331, 68.95861111111111)"
2016fnl,0.0163,"[0.0163, 0.016328]",[73],3.3,7.487667,32.893750,MRK 950,"(7.487499999999999, 32.893677777777775)"
2022gri,0.028,[0.028],[73],3.3,109.586643,33.994853,Unknown!,N/A
2022dsb,0.023,"[0.023, 0.0234]",[73],3.3,235.590633,-22.670580,Unknown!,N/A
2019azh,0.02199999988079071,"[0.02199999988079071, 0.022, 0.0222, 0.022346]",[73],3.3,123.320627,22.648301,KUG 0810+227,"(123.32083333333331, 22.648333333333333)"
2020wey,0.02738,"[0.02738, 0.02741]",[73],3.3,136.357958,61.802538,Unknown!,N/A


In [14]:
first_row = cands_df.iloc[0]

first_row

redshift                  0.01124
confidence                    3.1
ra                     228.847167
dec                     55.517108
Host info in OTTER?          True
Name: NGC 5905, dtype: object